# HW3 - Neural Network Model of BTS Data

## 1. Problem Definition

Goal: predict whether a flight's **departure will be delayed**, using neural network.

Features used(the same three as in HW2):
- `FLIGHT_DIRECTION`: general compass direction from origin state to destination state (0 = same state, 1 = Northwest, 2 = Southwest, 3 = Northeast, 4 = Southeast)
- `CRS_DEP_TIME`: scheduled departure time (continuous)
- `DAY_OF_MONTH`: day of the month (continuous)

## 2. Setup

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

SEED = 0
BATCH_SIZE = 1024
LEARNING_RATE = 3e-3
EPOCHS = 60

## 3. Import the Data

In [ ]:
import pandas as pd

df = pd.read_csv("DelayorNot_processed_all.csv")

## 4. Data Processing for Logistic regression
(It is the same as HW2)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

y = (df["DEP_DELAY_NEW"] > 0).astype(int)
X = df[["FLIGHT_DIRECTION", "CRS_DEP_TIME", "DAY_OF_MONTH"]]
X = pd.get_dummies(X, columns=["FLIGHT_DIRECTION"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)

continuous_cols = ["CRS_DEP_TIME", "DAY_OF_MONTH"]
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test_scaled[continuous_cols] = scaler.transform(X_test[continuous_cols])


## 5. Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(max_iter=200)
logreg.fit(X_train_scaled, y_train)

## 6. Neural Network (PyTorch)

**Data:** the standardized training data from above is split again into **75% train / 25% validation**, so that overall it is 60% train / 20% validation / 20% test. 

**Architecture:** 7 inputs -> `Linear(7, 64)` + ReLU -> `Linear(64, 32)` + ReLU -> `Linear(32, 16)` + ReLU -> `Linear(16, 1)`, so there are three hidden layers. The single output is the logit of "delayed". 

**Loss function:** binary cross-entropy loss (`BCEWithLogitsLoss`) and the Adam optimizer.

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_scaled, y_train, test_size=0.25, random_state=0, stratify=y_train
)

def to_tensor(X):
    return torch.tensor(X.to_numpy(dtype="float32"))

X_tr_t, X_val_t, X_test_t = to_tensor(X_tr), to_tensor(X_val), to_tensor(X_test_scaled)
y_tr_t = torch.tensor(y_tr.to_numpy(), dtype=torch.float32)
y_val_t = torch.tensor(y_val.to_numpy(), dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=BATCH_SIZE, shuffle=True)

print("train:", len(X_tr), " validation:", len(X_val), " test:", len(X_test_scaled))

In [ ]:
class DelayNet(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.fc1 = nn.Linear(n_features, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 16)
        self.fc4 = nn.Linear(16, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.fc4(x)
        return x.squeeze(1)


torch.manual_seed(SEED)
net = DelayNet(X_tr_t.shape[1])

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=LEARNING_RATE)

In [ ]:
history = []
for epoch in range(1, EPOCHS + 1):
    running_loss = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(net(xb), yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(xb)
    train_loss = running_loss / len(train_loader.dataset)

    with torch.no_grad():
        val_loss = loss_fn(net(X_val_t), y_val_t).item()

    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
    #print(f"epoch {epoch:2d} | train loss {train_loss:.4f} | val loss {val_loss:.4f}")

history = pd.DataFrame(history)

### Training curve

In [ ]:
plt.plot(history["epoch"], history["train_loss"], label="train loss")
plt.plot(history["epoch"], history["val_loss"], label="validation loss")
plt.xlabel("epoch")
plt.ylabel("binary cross-entropy loss")
plt.title("Training curve")
plt.legend()
plt.show()

## 7. Test Set Results

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score

with torch.no_grad():
    nn_prob = torch.sigmoid(net(X_test_t)).numpy()

predictions = {
    "Logistic Regression": (logreg.predict(X_test_scaled), logreg.predict_proba(X_test_scaled)[:, 1]),
    "Neural network": ((nn_prob >= 0.5).astype(int), nn_prob),
}

rows = [{"Model": "Always predict on-time", "Test error rate": y_test.mean(), "Test AUC": 0.5}]
for name, (y_pred, y_prob) in predictions.items():
    error_rate = 1 - accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    rows.append({"Model": name, "Test error rate": error_rate, "Test AUC": auc})

results = pd.DataFrame(rows)
results.round(4)

In [ ]:
plt.plot(results["Model"], results["Test error rate"], marker="o", color="dimgray")
for model_name, error in zip(results["Model"], results["Test error rate"]):
    plt.annotate(f"{error:.4f}", (model_name, error), textcoords="offset points", xytext=(0, 8), ha="center")
plt.ylabel("test error rate")
plt.title("Test error rate")
plt.ylim(0, results["Test error rate"].max() * 1.15)
plt.xticks(rotation=15)
plt.show()

## 8. Discussion

The neural network has a lower test error rate than logistic regression (0.3597 vs. 0.3925), and its AUC is also higher (0.6561 vs. 0.6104). Logistic regression can only learn a linear effect of the scheduled departure time and the day of the month, whereas the neural network can learn curved relationships. This suggests that the data contains some nonlinear patterns that the network was able to capture. However, the improvement is limited because the input features carry too little information. To predict whether a flight will be delayed, we therefore need to bring in more features that are known before the flight departs, such as the temperature and wind speed at the departure airport, which are likely to be more relevant. Without more informative features, the predictions will stay only about 4 percentage points better than always predicting on-time.


[1] Code was debugged using Claude Sonnet 5.5 (via Claude Code), October 5-6, 2026, Anthropic, https://claude.com/claude-code. The same tool was used to run small experiments on design choices (for example how many layers to use), to help my coding for neural network, and to polish the language.